# Create Virtual Icechunk Store from FVCOM GOM3 NetCDF3 Files

Uses VirtualiZarr to open each of the 468 uncompressed NetCDF3 files on S3 as a virtual dataset,
subchunks variables with vertical dimensions to 1 layer per chunk, then appends them all into
a single Icechunk repository stored on S3.

**Pipeline:**
1. Build a subchunked reference template from **file 0 only** (kerchunk + subchunk — one-time cost)
2. For every other file: read **8 bytes** from the NetCDF3 header to get `numrecs` (= ntime);
   compute time values from a running cursor (`t_current += ntime * dt`) — zero kerchunk calls
3. Clone data refs by URL substitution; encode time values inline as base64
4. Convert each file's refs to a VirtualiZarr virtual dataset and append to Icechunk

**Speed-up vs naive approach:** eliminates 467 `NetCDF3ToZarr` header scans — each file requires
only an 8-byte S3 read instead of scanning the entire NetCDF3 header.

In [ ]:
import dataclasses
import json
import os
import struct

import cftime
import numpy as np
import s3fs
import xarray as xr
import icechunk
from kerchunk.netCDF3 import NetCDF3ToZarr
from kerchunk.utils import subchunk
from obstore.store import S3Store
from obspec_utils.registry import ObjectStoreRegistry
from pathlib import Path
from virtualizarr.manifests import ManifestStore, ChunkManifest, ManifestArray
from virtualizarr.parsers.kerchunk.translator import manifestgroup_from_kerchunk_refs
from dotenv import load_dotenv

In [ ]:
load_dotenv(os.path.expanduser('~/dotenv/umassd-fvcom.yml'))

In [ ]:
# List source NetCDF3 files (public, anonymous access)
fs = s3fs.S3FileSystem(anon=True)
flist = sorted(fs.glob('umassd-fvcom/gom3/hindcast/*.nc'))
flist = [f's3://{f}' for f in flist]
print(len(flist), 'files')
print(flist[0])
print(flist[-1])

In [ ]:
SKIP_VARS = ['Itime', 'Itime2', 'Times', 'file_date', 'iint', 'nprocs']

# Variables with vertical dimensions that need subchunking
SIGLEV_VARS = ['kh', 'km', 'kq', 'l', 'omega', 'q2', 'q2l', 'siglev']
SIGLAY_VARS = ['salinity', 'siglay', 'temp', 'u', 'v', 'ww']
NLEV = 46  # siglev layers
NLAY = 45  # siglay layers

## Create Icechunk repository on S3

The `VirtualChunkContainer` tells Icechunk that virtual chunks prefixed with
`s3://umassd-fvcom/` live in the public bucket and should be fetched anonymously.
The repository itself is written using the credentials from `~/dotenv/umassd-fvcom.yml`.

In [ ]:
bucket = 'umassd-fvcom'
prefix = 'gom3/hindcast/icechunk/gom3.icechunk'
region = 'us-east-1'

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix='s3://umassd-fvcom/',
        store=icechunk.s3_store(region=region, anonymous=True),
    ),
)

storage = icechunk.s3_storage(
    bucket=bucket,
    prefix=prefix,
    region=region,
    access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'],
)

repo = icechunk.Repository.open_or_create(storage, config)

## Build reference template from file 0, then iterate over all files

All 468 FVCOM files share identical structure and byte offsets within each time record.
We parse file 0 once (with subchunking) to get fully-subchunked data refs, then for every
other file we only need a fast header read (~33 MB) to get that file's time coordinate.
All data variable chunk refs are reused via URL substitution — no Coiled or Dask needed.

In [ ]:
# ── Build subchunked reference template from file 0 ───────────────────────────
url_0 = flist[0]
print(f'Building template from {url_0} ...')

refs_0 = NetCDF3ToZarr(url_0, inline_threshold=0, storage_options={'anon': True}).translate()

# Subchunk vertical variables to 1 layer per chunk.
# subchunk() unwraps the dict on the first call; subsequent calls work on the flat dict.
flat0 = refs_0
for v in SIGLEV_VARS:
    flat0 = subchunk(flat0, variable=v, factor=NLEV)
for v in SIGLAY_VARS:
    flat0 = subchunk(flat0, variable=v, factor=NLAY)
# flat0 is now the inner flat dict (not wrapped in {'version': 1, 'refs': ...})

# Number of time steps in file 0 — used to identify time-varying variables
ntime_0 = json.loads(flat0['time/.zarray'])['shape'][0]

# Separate time refs from data refs.
# Time refs vary per file (each month has a different number of time steps).
# Data refs are reused for every file via URL substitution.
flat0_data = {k: v for k, v in flat0.items() if not k.startswith('time')}
print(f'  ntime_0:  {ntime_0}')
print(f'  Template data refs: {len(flat0_data):,} keys (broadcast to Coiled workers)')

In [ ]:
# ── Extract time metadata + build template virtual dataset (both once) ────────
# Two tiny byte-range reads give us t0 and dt; then we build the template vds
# from the subchunked flat0_data — this is the only call to manifestgroup_from_kerchunk_refs.

time_zattrs   = json.loads(flat0['time/.zattrs'])
time_units    = time_zattrs['units']
time_calendar = time_zattrs.get('calendar', 'standard')
_be_dtype     = np.dtype(json.loads(flat0['time/.zarray'])['dtype']).newbyteorder('>')

ref_t0, ref_t1 = flat0['time/0'], flat0['time/1']
with fs.open(url_0, 'rb') as _f:
    _f.seek(ref_t0[1]); t0_val = np.frombuffer(_f.read(ref_t0[2]), dtype=_be_dtype)[0]
    _f.seek(ref_t1[1]); t1_val = np.frombuffer(_f.read(ref_t1[2]), dtype=_be_dtype)[0]
dt_val = t1_val - t0_val
print(f'Time units={time_units}  t0={cftime.num2date(t0_val, time_units, time_calendar)}  dt={dt_val*24:.4f}h')

# Build the ManifestGroup and template virtual dataset once from flat0_data.
# All subsequent files are handled via rename_paths — no re-parsing needed.
mg       = manifestgroup_from_kerchunk_refs({'version': 1, 'refs': flat0_data},
                                             skip_variables=SKIP_VARS,
                                             fs_root=Path.cwd().as_uri())
_store   = S3Store.from_url(f's3://{bucket}', config={'skip_signature': True, 'region': region})
registry = ObjectStoreRegistry({f's3://{bucket}': _store})
ms       = ManifestStore(group=mg, registry=registry)
template_vds = ms.to_virtual_dataset(loadable_variables=[], decode_times=False)

# Identify which variables have time as their first dimension
time_var_names = {name for name, var in template_vds.data_vars.items()
                  if var.dims and var.dims[0] == 'time'}
print(f'Template vds ready  ({len(template_vds.data_vars)} vars, {len(time_var_names)} time-varying)')

In [ ]:
def get_ntime_from_header(url, fs):
    """
    Read 8 bytes from a NetCDF3 file to get numrecs (= number of time steps).

    NetCDF-3 layout:
      bytes 0-3 : magic  (b'CDF\\x01' classic  or  b'CDF\\x02' 64-bit offset)
      bytes 4-7 : numrecs  (big-endian int32)
    """
    with fs.open(url, 'rb') as f:
        hdr = f.read(8)
    assert hdr[:3] == b'CDF', f'Not a NetCDF3 file: {url}'
    return struct.unpack('>i', hdr[4:8])[0]


def clone_vds(url, ntime_n, step_count, template_vds, url_0, ntime_0, time_var_names,
              t0_val, dt_val, time_units, time_calendar):
    """
    Clone the template virtual dataset for a new file using rename_paths.

    - Substitutes url_0 → url in every ChunkManifest via rename_paths (numpy, no JSON parsing).
    - Trims the time dimension when ntime_n < ntime_0 using ChunkManifest.from_arrays.
    - Time values computed as t0_val + (step_count + i) * dt_val — integer step counting
      avoids floating-point drift that would break uniform spacing across 468 files.
    """
    new_vars = {}
    for name, var in template_vds.data_vars.items():
        if not hasattr(var.data, 'manifest'):
            new_vars[name] = var
            continue

        ma = var.data
        new_manifest = ma.manifest.rename_paths(lambda p: p.replace(url_0, url))

        if name in time_var_names and ntime_n < ntime_0:
            m = new_manifest
            new_manifest = ChunkManifest.from_arrays(
                paths=m._paths[:ntime_n],
                offsets=m._offsets[:ntime_n],
                lengths=m._lengths[:ntime_n],
            )
            new_meta = dataclasses.replace(ma.metadata, shape=(ntime_n,) + ma.metadata.shape[1:])
        else:
            new_meta = ma.metadata

        new_vars[name] = xr.Variable(var.dims,
                                     ManifestArray(metadata=new_meta, chunkmanifest=new_manifest),
                                     var.attrs)

    # Integer step counting: t0_val + (step_count + i) * dt_val avoids accumulated
    # floating-point error that would arise from repeatedly adding ntime_n * dt_val.
    raw = t0_val + (step_count + np.arange(ntime_n)) * dt_val
    time_coord = xr.Variable('time', cftime.num2date(raw, time_units, time_calendar))
    return xr.Dataset(new_vars, coords={'time': time_coord}, attrs=template_vds.attrs)

In [ ]:
%%time
session = repo.writable_session('main')

step_count = 0   # integer: total time steps written so far (no floating-point drift)

for i, f in enumerate(flist):
    ntime_n = ntime_0 if f == url_0 else get_ntime_from_header(f, fs)

    vds = clone_vds(f, ntime_n, step_count, template_vds, url_0, ntime_0,
                    time_var_names, t0_val, dt_val, time_units, time_calendar)

    if i == 0:
        vds.vz.to_icechunk(session.store)
    else:
        vds.vz.to_icechunk(session.store, append_dim='time')

    step_count += ntime_n

    if (i + 1) % 50 == 0 or i == 0:
        print(f'  [{i+1}/{len(flist)}] {f.split("/")[-1]}  ntime={ntime_n}  total_steps={step_count}')

snapshot_id = session.commit(
    'FVCOM GOM3 hindcast — 468 NetCDF3 files, subchunked 1 layer per vertical chunk'
)
print(f'Snapshot ID: {snapshot_id}')

## Verify the store is readable

In [ ]:
credentials = icechunk.containers_credentials({
    's3://umassd-fvcom/': icechunk.s3_credentials(anonymous=True)
})
repo_ro = icechunk.Repository.open(
    storage,
    config,
    authorize_virtual_chunk_access=credentials,
)
session_ro = repo_ro.readonly_session('main')
ds = xr.open_zarr(session_ro.store, consolidated=False, chunks=None)
ds